In [1]:
import os
# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [3]:
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [4]:
!pip install transformers torch --quiet

In [5]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')  # Suppress unnecessary warnings
import torch
# Choose the Qwen 2.5 1.5B Instruct model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

Model loaded successfully.
Device map: Single device


In [7]:
def generate_reply(chat_history, user_message, max_new_tokens=256):

    # Safety: handle empty input
    if not user_message.strip():
        return chat_history, "Please enter a valid message."

    # Add user message to history
    chat_history.append({"role": "user", "content": user_message})

    # Limit history (prevents memory overflow)
    chat_history = chat_history[-10:]

    # Create prompt using Qwen chat template
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize input
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,   # slightly improved
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only new tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    # Decode response
    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # Fallback if empty response
    if not assistant_reply:
        assistant_reply = "I'm not sure how to respond. Could you rephrase?"

    # Add assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})

    return chat_history, assistant_reply


In [8]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    # Initialize chat history
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]

    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue

        try:
            # Generate reply
            chat_history, assistant_reply = generate_reply(chat_history, user_input)

            # Fallback if empty
            if not assistant_reply.strip():
                assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."

        except Exception as e:   # ✅ small safety improvement
            assistant_reply = "Oops! Something went wrong. Please try again."

        print("Chatbot:", assistant_reply)


run_chatbot()


Chatbot: Hello! I am your AI assistant. How can I help you today?
User: What is Machine Learning?
Chatbot: Machine learning is a subset of artificial intelligence that involves using algorithms to enable computers to learn from data without being explicitly programmed. It allows systems to improve their performance on specific tasks over time as they "learn" from experience. The process typically includes three main steps: 
1. **Data Collection**: Gathering relevant information or input.
2. **Model Training**: Using the collected data to train models that can make predictions or decisions based on new inputs.
3. **Evaluation and Optimization**: Assessing how well the model performs and adjusting it if necessary.

Machine learning has applications in various fields such as image recognition, natural language processing, recommendation systems, and autonomous vehicles. Its ability to adapt through interaction with its environment makes it particularly useful for complex problems where tr